**DEPENDENCIES**

In [1]:
!pip install -q sentence-transformers faiss-cpu pandas PyPDF2 openpyxl

**CONFIGURATION**

In [2]:
import os

DEVICE = "cpu"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

EXCEL_FILES = ["TRANSCRIPT OF THERAPY SESSION Data set 2.xlsx"]
PDF_FILES = []

MAX_ROWS_PER_EXCEL = 500   #Due to RAM
CHUNK_SIZE_CHARS = 100
CHUNK_OVERLAP_CHARS = 10
TOP_K = 2

INDEX_DIR = "rag_index"
os.makedirs(INDEX_DIR, exist_ok=True)

FAISS_INDEX_PATH = os.path.join(INDEX_DIR, "index.faiss")
TEXTS_PATH = os.path.join(INDEX_DIR, "texts.json")
SOURCES_PATH = os.path.join(INDEX_DIR, "sources.json")

print("Files:", os.listdir("."))

Files: ['.config', 'therapy_session_transcript.xlsx', 'TRANSCRIPT OF THERAPY SESSION Data set 2.xlsx', 'rag_index', 'sample_data']


**CHUNKING**

In [3]:
import json
from typing import List, Dict
import pandas as pd
from PyPDF2 import PdfReader

def read_therapy_excel(path: str, max_rows: int = 1000) -> List[str]:
    df = pd.read_excel(path, nrows=max_rows)
    df = df.fillna("")
    rows = []
    for _, row in df.iterrows():
        parts = []
        for val in row.values:
            if isinstance(val, str) and val.strip():
                parts.append(val.strip())
        if parts:
            rows.append(" ".join(parts))
    return rows

def chunk_text(text: str, source_name: str) -> List[Dict[str, str]]:
    chunks = []
    n = len(text)
    start = 0
    while start < n:
        end = min(start + CHUNK_SIZE_CHARS, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append({"text": chunk, "source": source_name})
        start = end - CHUNK_OVERLAP_CHARS
        if start < 0:
            start = 0
    return chunks

print("Helpers ready")

Helpers ready


**INGESTION, INDEXING AND EMBEDDING**

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

def ingest_and_build_index():
    print("[INFO] ingest_and_build_index started")
    all_chunks = []

    # Only one Excel, 5 rows
    for xls in EXCEL_FILES:
        if not os.path.exists(xls):
            print("[WARN] Missing Excel:", xls)
            continue
        rows = read_therapy_excel(xls, max_rows=MAX_ROWS_PER_EXCEL)
        print(f"[INFO] {xls}: {len(rows)} rows used")
        for i, row_text in enumerate(rows):
            all_chunks.extend(
                chunk_text(row_text, source_name=f"EXCEL::{xls}::row_{i}")
            )

    print("[INFO] Total chunks:", len(all_chunks))
    if not all_chunks:
        print("[ERROR] No chunks, stopping")
        return

    texts = [c["text"] for c in all_chunks]
    sources = [c["source"] for c in all_chunks]

    print(f"[INFO] Loading model: {EMBED_MODEL_NAME} on {DEVICE}")
    model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

    emb_list = []
    batch_size = 1  #Change this, taken small due to RAM
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        print(f"[INFO] Embedding batch {i}-{i+len(batch)} / {len(texts)}")
        emb = model.encode(batch, convert_to_numpy=True, batch_size=len(batch))
        emb_list.append(emb)

    embeddings = np.vstack(emb_list).astype("float32")
    print("[INFO] Embeddings shape:", embeddings.shape)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    print("[INFO] FAISS index size:", index.ntotal)

    faiss.write_index(index, FAISS_INDEX_PATH)
    with open(TEXTS_PATH, "w", encoding="utf-8") as f:
        json.dump(texts, f, ensure_ascii=False)
    with open(SOURCES_PATH, "w", encoding="utf-8") as f:
        json.dump(sources, f, ensure_ascii=False)

    print("[INFO] Ingestion done")

ingest_and_build_index()



[INFO] ingest_and_build_index started
[INFO] TRANSCRIPT OF THERAPY SESSION Data set 2.xlsx: 3 rows used


**RETRIEVAL AND ASLO A DEMO**

In [ ]:
from typing import Any

class RAGRetriever:
    def __init__(self):
        print(f"[INFO] Loading model for retrieval: {EMBED_MODEL_NAME} on {DEVICE}")
        self.model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
        self.index = faiss.read_index(FAISS_INDEX_PATH)
        with open(TEXTS_PATH, "r", encoding="utf-8") as f:
            self.texts = json.load(f)
        with open(SOURCES_PATH, "r", encoding="utf-8") as f:
            self.sources = json.load(f)
        print(f"[INFO] Retriever ready with {len(self.texts)} chunks")

    def retrieve(self, query: str, top_k: int = TOP_K):
        q_emb = self.model.encode([query], convert_to_numpy=True).astype("float32")
        D, I = self.index.search(q_emb, top_k)
        results = []
        for rank, idx in enumerate(I[0]):
            results.append({
                "rank": rank + 1,
                "text": self.texts[idx],
                "source": self.sources[idx],
                "distance": float(D[0][rank]),
            })
        return results

retriever = RAGRetriever()
hits = retriever.retrieve("patient anxiety about relationship", top_k=3)
for h in hits:
    print("\nRank:", h["rank"], "| Dist:", f"{h['distance']:.4f}")
    print("Source:", h["source"])
    print("Text:", h["text"][:250], "...")